In [1]:
import os
import os.path
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm
import re

In [2]:
import os
import sys
from pathlib import Path

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "test"  # Change to "test" for final submission

# Set to True to rebuild indices from CSV (required on first run)
# Set to False to load cached indices (faster for subsequent runs)
FORCE_REBUILD_INDICES = False

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths
    DATA_PATH = Path("/kaggle/input/omnilex-data")
    MODEL_PATH = Path("/kaggle/input/llama-model")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/input/omnilex-indices")
    sys.path.insert(0, "/kaggle/input/omnilex-utils")
else:
    # Local development paths
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"
    INDEX_PATH = REPO_ROOT / "data" / "processed"

# CSV corpus files for index building
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"

# Index cache paths
LAWS_INDEX_PATH = INDEX_PATH / "laws_index.pkl"
COURTS_INDEX_PATH = INDEX_PATH / "courts_index.pkl"

# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Force rebuild indices: {FORCE_REBUILD_INDICES}")
print(f"\nCorpus files:")
print(f"  Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)" if LAWS_CSV.exists() else f"  Laws CSV: {LAWS_CSV} (NOT FOUND)")
print(f"  Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)" if COURTS_CSV.exists() else f"  Courts CSV: {COURTS_CSV} (NOT FOUND)")
print(f"\nIndex cache: {INDEX_PATH}")

Environment: Local
Dataset mode: test
Query file: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/test.csv
Validation mode: False
Force rebuild indices: False

Corpus files:
  Laws CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/laws_de.csv (73.0 MB)
  Courts CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/court_considerations.csv (2.28 GB)

Index cache: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

dense_model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True)
# reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation
# reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

In [4]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = {}
for citation, text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()):
    # if citation in court_consideration_d:
    #     court_consideration_d[citation] = court_consideration_d[citation] + '\n\n' + text
    # else:
    #     court_consideration_d[citation] = text
    court_consideration_d[citation] = text

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

test_df = pd.read_csv('../data/test_rewrite_001.csv')

_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}
    

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

print("data loaded")

data loaded


In [5]:
import dense_index
from dense_index import DenseIndex

print(dense_model.normalize_embeddings)
court_dense_index = DenseIndex(dense_model, "../data/processed/_dense_sparse_court", court_doc)
court_dense_index.info()

True
DenseIndex.embeddings:  (2107648, 1024)
[dense_index] documents.len: 1985178 parent_idx.len: 2107648


In [6]:
from sparse_index import SparseIndex

court_sparse_index = SparseIndex(dense_model, "../data/processed/_dense_sparse_court", court_doc)
court_sparse_index.load()

In [7]:
import citation_utils
import rerank_utils
import rrf
import reranker

RECALL_COUNT=1000
RERANK_COUNT=100
NN = 10

r = reranker.Reranker("/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3", "../ft_data/lora_reranker_output_30000_e1_0-145")

id_l = []
citation_l = []
for query_id, query_l in tqdm(test_dict.items(), total=len(test_dict)):
    ranked_l_l = []
    for query in query_l:
        court_sparse_search_l = court_dense_index.search(query, RECALL_COUNT)

    print(f"{query_id} court sparse search done.")

    second_layer = citation_utils.second_layer_citation_with_score(court_consideration_d, law_d, [(doc['citation'],1) for doc in court_sparse_search_l])

    law_hits = [{'citation':citation, 'text':law_d[citation]} for citation,score in second_layer if citation in law_d]
    court_hits = [{'citation':citation, 'text':court_consideration_d[citation]} for citation,score in second_layer if citation in court_consideration_d]

    print("court_hits.len:", len(court_hits), ", law_hits.len:", len(law_hits))

    # _court_l = []
    # _court_l.extend(court_sparse_search_l)
    # _court_l.extend(court_hits)
    # court_pairs = []
    # for hit in _court_l:
    #     court_pairs.append((query, hit['text']))
    # ret = r.compute_score(court_pairs)
    # court_rerank_l = []
    # for i, score in ret[:10]:
    #     court_rerank_l.append(_court_l[i])
    court_rerank_l = []
        
    # court_rerank_l = rerank_utils.rerank_by_dense_batch_chunked(reranker, query, _court_l, 10, 20, 384, 128)
    
    law_rerank_l = [c['citation'] for c in law_hits[:30]]
    
    # 去重
    citations = [doc['citation'] for doc in court_rerank_l]
    for c in law_rerank_l:
        citations.append(c)
    citations = list(set(citations))
    id_l.append(query_id)
    citation_l.append(';'.join(citations))
    print(query_id, len(citations))

result_df = pd.DataFrame({'query_id':id_l, 'predicted_citations':citation_l})
result_df.to_csv("../data/result.csv", index=False)

  2%|▎         | 1/40 [00:01<00:48,  1.24s/it]

test_001 court sparse search done.
court_hits.len: 88 , law_hits.len: 189
test_001 30


  5%|▌         | 2/40 [00:01<00:35,  1.08it/s]

test_002 court sparse search done.
court_hits.len: 103 , law_hits.len: 280
test_002 30


  8%|▊         | 3/40 [00:02<00:32,  1.16it/s]

...

test_038 court sparse search done.
court_hits.len: 242 , law_hits.len: 345
test_038 30


 98%|█████████▊| 39/40 [00:28<00:00,  1.44it/s]

test_039 court sparse search done.
court_hits.len: 282 , law_hits.len: 352
test_039 30


100%|██████████| 40/40 [00:28<00:00,  1.39it/s]

test_040 court sparse search done.
court_hits.len: 185 , law_hits.len: 238
test_040 30
